#### this is the script calculating the country area statistics with uncertainty.
The data used in this script contains points data from centroid of prediction pixels. Attributes includes original value, edge situation, inside or outside the study regions (exclude China). 
The country boundary data is from the global data lab. 

we only calculate the points inside our study region. different weights were given to the points according to its location. We aggragated the weighted points to countries.

The prediction coverage of the whole country was also calculated, together with the area of each class. we took only shifting cultivation above 100 ha into account during the statistic.

we use the producer accuracy to calculate the uncertainty of national shifting cultivation area.

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np

In [2]:
import geopandas as gpd
import pandas as pd

# --------------------------------------------------
# LOAD
# --------------------------------------------------
points = gpd.read_file(
    "/mnt/warehouse/shifting_cultivation/111_result/pred_points/weighted_points_result_ori.gpkg"
)

countries = gpd.read_file(
    "/home/wanting/Downloads/GDL Shapefiles V6.6/GDL Shapefiles V6.6 large.shp"
)

# --------------------------------------------------
# PREP COUNTRIES (CRITICAL SPEED STEP)
# --------------------------------------------------
countries = (
    countries[["iso_code", "country", "continent", "geometry"]]
    .dissolve(by="country", as_index=False)
)

# CRS match
points = points.to_crs(countries.crs)

# --------------------------------------------------
# FILTER + REDUCE COLUMNS
# --------------------------------------------------
points = points.loc[points["inside_mask"] == 1,
                    ["geometry", "orig_value", "edge_flag"]].copy()

points["orig_value"] = points["orig_value"].astype("int8")

# --------------------------------------------------
# WEIGHTS
# --------------------------------------------------
points["weight"] = points["edge_flag"].map({0: 1.0, 1: 0.5, 2: 0.25})

# --------------------------------------------------
# FAST SPATIAL JOIN
# --------------------------------------------------
points = gpd.sjoin(
    points,
    countries,
    how="left",
    predicate="within"
)

points = points.dropna(subset=["country"])

# --------------------------------------------------
# AGGREGATION
# --------------------------------------------------
country_class_area = (
    points
    .groupby(["country", "orig_value"], observed=True)["weight"]
    .sum()
    .reset_index()
    .rename(columns={"orig_value": "class_id", "weight": "area_km2"})
)

country_class_area_wide = (
    country_class_area
    .pivot(index="country", columns="class_id", values="area_km2")
    .fillna(0)
)

country_class_area_wide.columns = [
    f"class_{int(c)}_km2" for c in country_class_area_wide.columns
]

country_class_area_wide = country_class_area_wide.reset_index()


In [3]:
country_class_area_wide = country_class_area_wide.drop(
    columns=["geometry"], errors="ignore"
)

In [5]:
country_class_area_wide.to_csv("/mnt/warehouse/shifting_cultivation/111_result/output_file/country_class_weighted_area_km2.csv")

I can label the countries that have fully prediction or partially prediction.
1. fully or partially 
2. area of prediction
use the grid and countries layer to have this columns

In [4]:
import geopandas as gpd
import pandas as pd

# -----------------------------
# Load data
# -----------------------------
pred_grids = gpd.read_file(
    "/mnt/warehouse/shifting_cultivation/111_result/grids/intersect_grids_checked.gpkg"
)

countries_raw = gpd.read_file(
    "/home/wanting/Downloads/GDL Shapefiles V6.6/GDL Shapefiles V6.6 large.shp"
)

# -----------------------------
# Dissolve subnational → country
# -----------------------------
countries_country = (
    countries_raw
    .dissolve(by="country", as_index=False)
)

# -----------------------------
# Reproject
# -----------------------------
equal_area_crs = "ESRI:54009"  # Mollweide

pred_grids = pred_grids.to_crs(equal_area_crs)
countries_country = countries_country.to_crs(equal_area_crs)

# -----------------------------
# Prediction footprint
# -----------------------------
pred_footprint = pred_grids.dissolve()

# -----------------------------
# Country total area
# -----------------------------
countries_country["country_area_km2"] = (
    countries_country.geometry.area / 1e6
)

# -----------------------------
# Intersection → predicted area
# -----------------------------
country_pred_intersection = gpd.overlay(
    countries_country[["country", "geometry"]],
    pred_footprint,
    how="intersection"
)

country_pred_intersection["pred_area_km2"] = (
    country_pred_intersection.geometry.area / 1e6
)

pred_area_by_country = (
    country_pred_intersection
    .groupby("country")["pred_area_km2"]
    .sum()
    .reset_index()
)

# -----------------------------
# Coverage table
# -----------------------------
country_coverage = (
    countries_country
    .merge(pred_area_by_country, on="country", how="left")
)

country_coverage["pred_area_km2"] = (
    country_coverage["pred_area_km2"].fillna(0)
)

country_coverage["coverage_pct"] = (
    100 * country_coverage["pred_area_km2"] /
    country_coverage["country_area_km2"]
)

def coverage_label(pct, tol=99.0):
    if pct >= tol:
        return "Fully predicted"
    elif pct > 0:
        return "Partially predicted"
    else:
        return "Not predicted"

country_coverage["prediction_status"] = (
    country_coverage["coverage_pct"].apply(coverage_label)
)

# -----------------------------
# Merge into class-area table
# -----------------------------
country_class_area_wide = (
    country_class_area_wide
    .merge(
        country_coverage[
            ["country", "country_area_km2",
             "pred_area_km2", "coverage_pct",
             "prediction_status", "geometry"]
        ],
        on="country",
        how="left"
    )
)

In [5]:
country_class_area_wide = gpd.GeoDataFrame(
    country_class_area_wide,
    geometry="geometry",
    crs=equal_area_crs
)

In [6]:
country_class_area_wide.columns

Index(['country', 'class_0_km2', 'class_1_km2', 'class_2_km2', 'class_3_km2',
       'class_4_km2', 'country_area_km2', 'pred_area_km2', 'coverage_pct',
       'prediction_status', 'geometry'],
      dtype='object')

In [7]:
country_class_area_wide["class_1_km2"] = pd.to_numeric(
    country_class_area_wide["class_1_km2"], errors="coerce"
)

In [8]:
import numpy as np

UA_SC = 0.817
n_SC = 150
z = 1.96

def sc_ci_only(A_map):
    se = A_map * np.sqrt(UA_SC * (1 - UA_SC) / n_SC)
    ci = z * se
    return ci

# CI in km²
country_class_area_wide["sc_ci_km2"] = (
    country_class_area_wide["class_1_km2"]
    .apply(sc_ci_only)
)

# Convert to Mha
country_class_area_wide["sc_ci_Mha"] = (
    country_class_area_wide["sc_ci_km2"] / 10_000
)

In [9]:
# Convert to Mha
country_class_area_wide["sc_Mha"] = (
    country_class_area_wide["class_1_km2"] / 10_000
)

In [10]:
country_class_area_wide = country_class_area_wide.set_geometry("geometry")

In [7]:
# -----------------------------
# Save
# -----------------------------
output_gpkg = "/mnt/warehouse/shifting_cultivation/111_result/output_file/country_stat/country_class_area_with_coverage_uncertainty_new_over100ha_newUA.gpkg"
country_class_area_wide.to_file(output_gpkg,driver="GPKG")

In [8]:
country_class_area_wide = gpd.read_file("/mnt/warehouse/shifting_cultivation/111_result/output_file/country_stat/country_class_area_with_coverage_uncertainty_new_over100ha_newUA.gpkg")

In [9]:
country_class_area_wide.columns

Index(['Unnamed: 0', 'field_1', 'country', 'class_0_km2', 'class_1_km2',
       'class_2_km2', 'class_3_km2', 'class_4_km2', 'country_area_km2',
       'pred_area_km2', 'coverage_pct', 'prediction_status', 'sc_ci_km2',
       'sc_ci_Mha', 'sc_Mha', 'continent', 'cropland_area', 'geometry'],
      dtype='object')

In [10]:
continent_sc_area = (
    country_class_area_wide
    .groupby("continent", as_index=False)
    .agg(
        sc_area_Mha=("class_1_km2", "sum")
    )
)

In [11]:
continent_sc_area

,continent,sc_area_Mha
0,Africa,937823.5
1,Aisa-Oceania,430057.5
2,America,408981.5


In [12]:
import numpy as np

UA_SC = 0.817
n_SC = 150
z = 1.96

In [13]:
class_cols = [
    "class_0_km2",
    "class_1_km2",
    "class_2_km2",
    "class_3_km2",
    "class_4_km2"
]

global_class_area = (
    country_class_area_wide[class_cols]
    .sum()
    .rename("area_km2")
    .reset_index()
    .rename(columns={"index": "class"})
)

# Convert to Mha
global_class_area["area_Mha"] = global_class_area["area_km2"] / 10_000

global_class_area

,class,area_km2,area_Mha
0,class_0_km2,9465050.25,946.505025
1,class_1_km2,1776862.50,177.686250
2,class_2_km2,10357143.25,1035.714325
3,class_3_km2,9827154.25,982.715425
4,class_4_km2,4337030.00,433.703000


In [14]:
total_area_km2 = global_class_area["area_km2"].sum()

In [15]:
import numpy as np
import pandas as pd

def area_ci_from_UA(
    area_km2,
    user_accuracy,
    n_samples,
    z=1.96
):
    """
    Compute 95% confidence interval for map-derived area
    using User's Accuracy (binomial approximation).

    Parameters
    ----------
    area_km2 : float
        Map-derived area in km²
    user_accuracy : float
        User's Accuracy for the class (0–1)
    n_samples : int
        Number of validation samples for that class
    z : float
        z-score (1.96 for 95% CI)

    Returns
    -------
    dict
        Area, SE, CI (km² and Mha)
    """

    se_km2 = area_km2 * np.sqrt(
        user_accuracy * (1 - user_accuracy) / n_samples
    )

    ci_km2 = z * se_km2

    return {
        "area_km2": area_km2,
        "area_Mha": area_km2 / 10_000,
        "se_km2": se_km2,
        "ci_km2": ci_km2,
        "ci_Mha": ci_km2 / 10_000,
        "lower_km2": area_km2 - ci_km2,
        "upper_km2": area_km2 + ci_km2,
    }


In [19]:
UA_SC = 0.817
n_SC = 150
sc_area_km2 = 1776862.50

result = area_ci_from_UA(
    area_km2=sc_area_km2,
    user_accuracy=UA_SC,
    n_samples=n_SC
)

pd.Series(result).round(2)


area_km2     1776862.50
area_Mha         177.69
se_km2         56097.66
ci_km2        109951.42
ci_Mha            11.00
lower_km2    1666911.08
upper_km2    1886813.92
dtype: float64